# PTC Yard Allocation as a MILP

This notebook reformulates the `OptimisedV4` yard-allocation logic as a mixed-integer linear program (MILP) using the same JuMP + HiGHS workflow and boilerplate style as the class MILP notebook.

What this notebook does:
- Reads `State.csv` and `Slots.csv`
- Segments the incoming container by ETA
- Generates the same physically feasible 20ft slots or 40ft slot-pairs as `OptimisedV4`
- Builds a binary assignment MILP over those feasible candidates
- Solves the model using HiGHS and reports the selected placement


## 1. Setup

We use the same package setup style as the class MILP notebook:
- `LinearAlgebra`, `Random`, `SparseArrays`
- `JuMP` as the modeling language
- `HiGHS` as the solver
- `Format` for display formatting
- `CSV`, `DataFrames` for the yard data


In [2]:
## SETS UP THE JULIA PROJECT ENVIRONMENT ##
# ═══════════════════════════════════════ #

import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate();
using Statistics


  Activating project at `~/Desktop/SUTD/Y2/T4/40.011 Data and Business Analytics/Poh Tiong Choon/DBA PTC Multi Yard/Universal`


In [3]:
## IMPORTS THE NECESSARY JULIA PACKAGES ##
# ══════════════════════════════════════ #

using LinearAlgebra, Random, SparseArrays, JuMP, HiGHS, Format
using CSV, DataFrames


## 2. Read the PTC Yard Data

The next cell reads the current yard state (`State.csv`) and slot master (`Slots.csv`).


In [31]:
## READS THE PTC YARD DATA ##
# ═════════════════════════ #

# ==============================
# YARD SELECTION (EDIT HERE)
# ==============================
selected_yard = "One"   # One, Two, Three, M, R

valid_yards = ["One", "Two", "Three", "M", "R"]
if !(selected_yard in valid_yards)
    error("Invalid yard. Choose from $(valid_yards)")
end

yards_folder = joinpath(@__DIR__, "Yards")
state_file = joinpath(yards_folder, "$(selected_yard)State.csv")
slots_file = joinpath(yards_folder, "$(selected_yard)Slots.csv")

if !isfile(state_file)
    error("State file not found: " * state_file)
end

if !isfile(slots_file)
    error("Slots file not found: " * slots_file)
end

# ------------------------------
# HELPERS FOR CLEANING
# ------------------------------
is_blank_cell(x) = ismissing(x) || strip(string(x)) == ""

function remove_fully_empty_rows(df::DataFrame)
    filter(row -> !all(is_blank_cell, row), df)
end

function remove_rows_missing_key(df::DataFrame, key::Symbol)
    if !(key in names(df))
        return df
    end
    filter(row -> !is_blank_cell(row[key]), df)
end

# ------------------------------
# LOAD RAW CSV DATA
# ------------------------------
container_data = CSV.File(state_file) |> DataFrame
slot_data = CSV.File(slots_file) |> DataFrame

# ------------------------------
# REMOVE EMPTY / JUNK ROWS
# ------------------------------
container_data = remove_fully_empty_rows(container_data)
slot_data = remove_fully_empty_rows(slot_data)

# Use key columns to remove rows that are effectively blank records
if :MovementId in names(container_data)
    container_data = remove_rows_missing_key(container_data, :MovementId)
elseif :Location in names(container_data)
    container_data = remove_rows_missing_key(container_data, :Location)
end

if :Location in names(slot_data)
    slot_data = remove_rows_missing_key(slot_data, :Location)
end

# ------------------------------
# CLEAN ETA COLUMN
# ------------------------------
if "ETA_hours" in names(container_data)
    container_data = filter(row -> !is_blank_cell(row.ETA_hours), container_data)
    container_data.ETA_hours = Float64.(container_data.ETA_hours)
end

println("===== SELECTED YARD =====")
println("Yard       = ", selected_yard)
println("State file = ", state_file)
println("Slots file = ", slots_file)
println("State rows = ", nrow(container_data))
println("Slot rows  = ", nrow(slot_data))

# Optional diagnostics
println("\n===== DATA CHECK =====")
if :MovementId in names(container_data)
    println("Missing MovementId in state = ", count(x -> is_blank_cell(x), container_data.MovementId))
end
if :Location in names(container_data)
    println("Missing Location in state   = ", count(x -> is_blank_cell(x), container_data.Location))
end
if :Location in names(slot_data)
    println("Missing Location in slots   = ", count(x -> is_blank_cell(x), slot_data.Location))
end
if :Tier in names(slot_data)
    println("Missing Tier in slots       = ", count(ismissing, slot_data.Tier))
end
if :Block in names(slot_data)
    println("Missing Block in slots      = ", count(ismissing, slot_data.Block))
end

===== SELECTED YARD =====
Yard       = One
State file = /Users/sheil/Desktop/SUTD/Y2/T4/40.011 Data and Business Analytics/Poh Tiong Choon/DBA PTC Multi Yard/Universal/Yards/OneState.csv
Slots file = /Users/sheil/Desktop/SUTD/Y2/T4/40.011 Data and Business Analytics/Poh Tiong Choon/DBA PTC Multi Yard/Universal/Yards/OneSlots.csv
State rows = 432
Slot rows  = 3066

===== DATA CHECK =====


## 3. Incoming Container Inputs

These are the same user inputs as in `OptimisedV4`.


In [32]:
## DEFINES THE INCOMING CONTAINER INPUTS ##
# ═════════════════════════════════════ #

# ==============================
# USER INPUT (EDIT HERE)
# ==============================
incoming_eta   = 300
incoming_width = 1      # 1 = 20ft, 2 = 40ft
incoming_laden = 1      # 0 = Empty, 1 = Laden
incomming_id   = 17

# ==============================
# VALIDATION
# ==============================
if !(incoming_width in [1, 2])
    error("incoming_width must be 1 or 2")
end

if !(incoming_laden in [0, 1])
    error("incoming_laden must be 0 or 1")
end

println("===== INPUT =====")
println("ETA        = ", incoming_eta)
println("Width      = ", incoming_width)
println("Laden      = ", incoming_laden)
println("CustomerId = ", incomming_id)



===== INPUT =====
ETA        = 300
Width      = 1
Laden      = 1
CustomerId = 17


## 4. Yard Helper Functions

These helper functions preserve the same structure and physical logic from `OptimisedV4`:
- parse a location like `30A5`
- define the 40ft partner as the next block, same column, same tier
- segment the incoming container by ETA
- enforce support / stack-continuity feasibility


In [33]:


## DEFINES BLOCK ORDER, LOCATION PARSING, AND ADJACENT 40FT PAIRING ##
# ══════════════════════════════════════════════════════════════════ #

# Get ordered unique blocks from Slots.csv
# This preserves the physical block order as it appears in the file.
block_order = unique(slot_data.Block)

# Map each block to its position in that ordered list
block_positions = Dict(block => i for (i, block) in enumerate(block_order))

# Parse location like:
#   30A5   -> block="30",  col="A", tier=5
#   40SA4  -> block="40S", col="A", tier=4
#   67SB6  -> block="67S", col="B", tier=6
function parse_location(loc::AbstractString)
    m = match(r"^(.*)([A-Z])(\d+)$", strip(loc))
    if m === nothing
        return nothing
    end
    block = m.captures[1]
    col   = m.captures[2]
    tier  = parse(Int, m.captures[3])
    return (block = block, col = col, tier = tier)
end

# Returns the partner location for a 40ft container:
# same column, same tier, next block in Slots.csv order.
# Example:
#   30A5  -> 31A5
#   40SA4 -> 41SA4
function adjacent_location(loc::AbstractString)
    p = parse_location(loc)
    p === nothing && return missing

    current_block = p.block

    if !haskey(block_positions, current_block)
        return missing
    end

    next_pos = block_positions[current_block] + 1
    if next_pos > length(block_order)
        return missing
    end

    next_block = block_order[next_pos]
    return string(next_block, p.col, p.tier)
end

function eta_segment(eta::Real)
    return eta < 120 ? "low" : "high"
end

segment = eta_segment(incoming_eta)

println("\nIncoming ETA = ", incoming_eta)
println("Incoming width = ", incoming_width)
println("ETA segment = ", segment)
println("Incoming customer id = ", incomming_id)
println("Laden Status = ", incoming_laden)

# Quick lookup dictionaries.
occupied_lookup = Dict(row.Location => row.OccupiedNow for row in eachrow(slot_data))
tier_lookup     = Dict(row.Location => row.Tier for row in eachrow(slot_data))
cost_lookup     = Dict(row.Location => row.Cost for row in eachrow(slot_data))
adjcost_lookup  = Dict(row.Location => row.AdjCost for row in eachrow(slot_data))
block_lookup    = Dict(row.Location => string(row.Block) for row in eachrow(slot_data))

# Ordered block list from Slots.csv (duplicates removed, order preserved).
block_order = unique(string.(slot_data.Block))
block_positions = Dict(block => idx for (idx, block) in enumerate(block_order))

# Returns all lower locations in the same stack.
function lower_stack_locations(loc::AbstractString)
    p = parse_location(loc)
    p === nothing && return String[]
    if p.tier == 1
        return String[]
    end
    return [string(p.block, p.col, t) for t in 1:(p.tier - 1)]
end

function get_customer_containers(df::DataFrame, customer_id)
    rows = df[df.CustomerId .== customer_id, :]

    if nrow(rows) == 0
        println("Customer $(customer_id) has no containers in the yard.")
        return DataFrame()
    end

    result = rows[:, [:MovementId, :CustomerId, :ContainerSize, :WidthUnits, :Location, :ETA_hours, :ShiftCount]]
    println("Customer $(customer_id) has $(nrow(result)) container(s) in the yard.")
    return result
end

function get_customer_blocks(customer_rows::DataFrame)
    if nrow(customer_rows) == 0
        return String[]
    end
    locs = customer_rows.Location
    blocks = String[]
    for loc in locs
        if !ismissing(loc) && haskey(block_lookup, loc)
            push!(blocks, block_lookup[loc])
        end
    end
    return unique(blocks)
end

function nearest_block_distance(block::AbstractString, customer_blocks::Vector{String})
    isempty(customer_blocks) && return typemax(Int)
    !haskey(block_positions, block) && return typemax(Int)

    valid_customer_blocks = [b for b in customer_blocks if haskey(block_positions, b)]
    isempty(valid_customer_blocks) && return typemax(Int)

    return minimum(abs(block_positions[block] - block_positions[b]) for b in valid_customer_blocks)
end

# 20ft feasibility:
# - target slot must be free
# - all lower tiers below it must be occupied
function feasible_20ft(loc::AbstractString)
    !haskey(occupied_lookup, loc) && return false
    occupied_lookup[loc] != 0 && return false

    for lower_loc in lower_stack_locations(loc)
        if !haskey(occupied_lookup, lower_loc)
            return false
        end
        if occupied_lookup[lower_loc] != 1
            return false
        end
    end
    return true
end

# 40ft feasibility:
# - loc and partner must both exist and both be free
# - all lower tiers below both must be occupied
function feasible_40ft(loc::AbstractString)
    partner = adjacent_location(loc)

    ismissing(partner) && return false
    !haskey(occupied_lookup, loc) && return false
    !haskey(occupied_lookup, partner) && return false

    occupied_lookup[loc] != 0 && return false
    occupied_lookup[partner] != 0 && return false

    p1 = parse_location(loc)
    p2 = parse_location(partner)
    (p1 === nothing || p2 === nothing) && return false
    p1.tier != p2.tier && return false

    for t in 1:(p1.tier - 1)
        lower1 = string(p1.block, p1.col, t)
        lower2 = string(p2.block, p2.col, t)

        if !haskey(occupied_lookup, lower1) || !haskey(occupied_lookup, lower2)
            return false
        end
        if occupied_lookup[lower1] != 1 || occupied_lookup[lower2] != 1
            return false
        end
    end
    return true
end



Incoming ETA = 300
Incoming width = 1
ETA segment = high
Incoming customer id = 17
Laden Status = 1


feasible_40ft (generic function with 1 method)

## 5. Use Historical Data to Infer the Target Tier

## 6. Generate Feasible Candidates Using the Historical Target Tier and Customer Block Preference

In [34]:
## USES HISTORICAL DATA TO INFER THE TARGET TIER ##
# ═════════════════════════════════════════════════════ #

same_size_history = container_data[container_data.WidthUnits .== incoming_width, :]

history_pool =
    nrow(same_size_history) >= 5 ? same_size_history : container_data

threshold = 1
min_tier = minimum(slot_data.Tier)
max_tier = maximum(slot_data.Tier)

if nrow(history_pool) == 0
    println("\nNo historical containers found in State.csv.")
    println("Using fallback target tier.")

    # fallback rule
    if incoming_eta < 120
        target_tier = max_tier   # low ETA -> prefer higher
    else
        target_tier = min_tier   # high ETA -> prefer lower
    end

    avg_hist_tier = missing
    avg_reshifting = missing

    closest_containers = DataFrame()
    closest_with_tier = DataFrame()

else
    history_pool = copy(history_pool)
    history_pool.eta_diff = abs.(history_pool.ETA_hours .- incoming_eta)
    sorted_containers = sort(history_pool, :eta_diff)
    closest_containers = sorted_containers[1:min(10, nrow(sorted_containers)), :]

    println("\nClosest historical containers:")
    println(closest_containers[:, [:Location, :ETA_hours, :ShiftCount, :WidthUnits, :eta_diff]])

    closest_with_tier = DataFrame(
        Location   = closest_containers.Location,
        ETA_hours  = closest_containers.ETA_hours,
        ShiftCount = coalesce.(closest_containers.ShiftCount, 0),
        Tier = [
            begin
                row = slot_data[slot_data.Location .== loc, :]
                nrow(row) == 0 ? missing : row[1, :Tier]
            end
            for loc in closest_containers.Location
        ]
    )

    println("\nClosest historical containers with tiers:")
    println(closest_with_tier)

    valid_tiers = collect(skipmissing(closest_with_tier.Tier))

    if isempty(valid_tiers)
        println("\nNo valid historical tiers found. Using fallback target tier.")
        if incoming_eta < 120
            target_tier = max_tier
        else
            target_tier = min_tier
        end
        avg_hist_tier = missing
        avg_reshifting = mean(closest_with_tier.ShiftCount)
    else
        avg_hist_tier = mean(valid_tiers)
        avg_reshifting = mean(closest_with_tier.ShiftCount)

        target_tier = floor(Int, avg_hist_tier)

        if incoming_eta < 120
            # LOW ETA: if reshifting is high, push upward
            if avg_reshifting >= threshold
                target_tier += 1
            end
        else
            # HIGH ETA: if reshifting is high, push downward
            if avg_reshifting >= threshold
                target_tier -= 1
            end
        end

        target_tier = max(target_tier, min_tier)
        target_tier = min(target_tier, max_tier)
    end
end

println("\nAverage historical tier = ", avg_hist_tier)
println("Average reshifting = ", avg_reshifting)
println("Threshold = ", threshold)
println("Target tier = ", target_tier)

bigM = max_tier - min_tier + 1

# Preference rule:
# - exact target tier is always best
# - LOW ETA  (<120): if target tier unavailable, prefer the next HIGHER tier first, then lower tiers
# - HIGH ETA (>=120): if target tier unavailable, prefer the next LOWER tier first, then higher tiers
function tier_preference_score(tier::Int, target_tier::Int, incoming_eta::Real, bigM::Int)
    if incoming_eta < 120
        return tier >= target_tier ? (tier - target_tier) : (bigM + (target_tier - tier))
    else
        return tier <= target_tier ? (target_tier - tier) : (bigM + (tier - target_tier))
    end
end

customer_containers = get_customer_containers(container_data, incomming_id)
customer_blocks = get_customer_blocks(customer_containers)
use_customer_grouping = !isempty(customer_blocks)

println("\nCustomer grouping active = ", use_customer_grouping)
println("Customer blocks = ", customer_blocks)

println("\n===== Historical Analysis Summary =====")
println("Average historical tier      = ", avg_hist_tier)
println("Average historical shifting  = ", avg_reshifting)
println("Threshold                    = ", threshold)
println("Target tier                  = ", target_tier)


Closest historical containers:
10×5 DataFrame
 Row │ Location  ETA_hours  ShiftCount  WidthUnits  eta_diff 
     │ String7   Float64    Int64       Int64       Float64  
─────┼───────────────────────────────────────────────────────
   1 │ 48G1         302.03           1           1      2.03
   2 │ 56SB1        302.87           2           1      2.87
   3 │ 39G3         294.72           2           1      5.28
   4 │ 62H1         292.6            0           1      7.4
   5 │ 62H2         292.6            0           1      7.4
   6 │ 62H3         292.12           0           1      7.88
   7 │ 62H4         292.12           0           1      7.88
   8 │ 56SC4        276.6            1           1     23.4
   9 │ 67SB3        324.75           1           1     24.75
  10 │ 47H4         324.95           0           1     24.95

Closest historical containers with tiers:
10×4 DataFrame
 Row │ Location  ETA_hours  ShiftCount  Tier  
     │ String7   Float64    Int64       Int64 
─────┼──

In [35]:
## GENERATES THE FEASIBLE CANDIDATES FOR THE MILP ##
# ═══════════════════════════════════════════════ #

function customer_grouping_band(block_distance::Int)
    if block_distance == 0
        return "same"
    elseif block_distance <= 3
        return "near"
    elseif block_distance <= 5
        return "mid"
    elseif block_distance <= 10
        return "far"
    else
        return "veryfar"
    end
end

if incoming_width == 1

    candidate_locs         = String[]
    candidate_blocks       = String[]
    candidate_tiers        = Int[]
    candidate_costs        = Float64[]
    candidate_preferences  = Int[]
    candidate_tierdiffs    = Int[]
    candidate_blockdists   = Int[]
    candidate_scores       = Float64[]

    for loc in slot_data.Location
        if feasible_20ft(loc)
            pref = tier_preference_score(tier_lookup[loc], target_tier, incoming_eta, bigM)
            tierdiff = abs(tier_lookup[loc] - target_tier)
            block = block_lookup[loc]
            blockdist = use_customer_grouping ? nearest_block_distance(block, customer_blocks) : typemax(Int)

            push!(candidate_locs, loc)
            push!(candidate_blocks, block)
            push!(candidate_tiers, tier_lookup[loc])
            push!(candidate_costs, float(cost_lookup[loc]))
            push!(candidate_preferences, pref)
            push!(candidate_tierdiffs, tierdiff)
            push!(candidate_blockdists, blockdist)
            push!(candidate_scores, 1000.0 * pref + float(cost_lookup[loc]))
        end
    end

    candidates = DataFrame(
        Candidate       = 1:length(candidate_locs),
        Location        = candidate_locs,
        Block           = candidate_blocks,
        Tier            = candidate_tiers,
        Cost            = candidate_costs,
        PreferenceScore = candidate_preferences,
        TierDiff        = candidate_tierdiffs,
        BlockDistance   = candidate_blockdists,
        Score           = candidate_scores
    )

    if use_customer_grouping
        selected_band = "fallback"

        same_block_exact = candidates[(candidates.BlockDistance .== 0) .& (candidates.PreferenceScore .== 0), :]
        if nrow(same_block_exact) > 0
            selected_band = "same block exact tier"
            same_block_exact.Score = same_block_exact.Cost
            candidates = same_block_exact
        else
            near_candidates = candidates[candidates.BlockDistance .<= 3, :]
            if nrow(near_candidates) > 0
                selected_band = "within 3 blocks"
                near_candidates.Score = 1000.0 .* near_candidates.PreferenceScore .+ 10.0 .* near_candidates.BlockDistance .+ near_candidates.Cost
                candidates = near_candidates
            else
                mid_candidates = candidates[(candidates.BlockDistance .> 3) .& (candidates.BlockDistance .<= 5) .& (candidates.TierDiff .<= 1), :]
                if nrow(mid_candidates) > 0
                    selected_band = "3 to 5 blocks (<= 1 tier sacrifice)"
                    mid_candidates.Score = 1000.0 .* mid_candidates.BlockDistance .+ 100.0 .* mid_candidates.TierDiff .+ mid_candidates.Cost
                    candidates = mid_candidates
                else
                    far_candidates = candidates[(candidates.BlockDistance .> 5) .& (candidates.BlockDistance .<= 10) .& (candidates.TierDiff .<= 2), :]
                    if nrow(far_candidates) > 0
                        selected_band = "5 to 10 blocks (<= 2 tier sacrifice)"
                        far_candidates.Score = 1000.0 .* far_candidates.BlockDistance .+ 100.0 .* far_candidates.TierDiff .+ far_candidates.Cost
                        candidates = far_candidates
                    else
                        veryfar_candidates = candidates[(candidates.BlockDistance .> 10) .& (candidates.TierDiff .== 0), :]
                        if nrow(veryfar_candidates) > 0
                            selected_band = "10+ blocks (exact tier only)"
                            veryfar_candidates.Score = 1000.0 .* veryfar_candidates.BlockDistance .+ veryfar_candidates.Cost
                            candidates = veryfar_candidates
                        else
                            selected_band = "historic tier fallback"
                        end
                    end
                end
            end
        end

        println("\nCustomer grouping search band used: ", selected_band)
    end

    println("\nFeasible 20ft candidates:")
    println(sort(candidates, [:Score, :BlockDistance, :PreferenceScore, :Cost]))

elseif incoming_width == 2

    candidate_locs         = String[]
    candidate_partners     = String[]
    candidate_blocks       = String[]
    candidate_partnerblks  = String[]
    candidate_tiers        = Int[]
    candidate_adjcosts     = Float64[]
    candidate_preferences  = Int[]
    candidate_tierdiffs    = Int[]
    candidate_blockdists   = Int[]
    candidate_scores       = Float64[]

    seen_pairs = Set{Tuple{String,String}}()

    for loc in slot_data.Location
        partner = adjacent_location(loc)

        if ismissing(partner)
            continue
        end
        if !haskey(adjcost_lookup, loc)
            continue
        end
        if ismissing(adjcost_lookup[loc]) || adjcost_lookup[loc] == 9999
            continue
        end
        if !feasible_40ft(loc)
            continue
        end

        pair_key = (loc, partner)
        if pair_key in seen_pairs
            continue
        end
        push!(seen_pairs, pair_key)

        pref = tier_preference_score(tier_lookup[loc], target_tier, incoming_eta, bigM)
        tierdiff = abs(tier_lookup[loc] - target_tier)

        block1 = block_lookup[loc]
        block2 = block_lookup[partner]
        blockdist = if use_customer_grouping
            min(nearest_block_distance(block1, customer_blocks), nearest_block_distance(block2, customer_blocks))
        else
            typemax(Int)
        end

        push!(candidate_locs, loc)
        push!(candidate_partners, partner)
        push!(candidate_blocks, block1)
        push!(candidate_partnerblks, block2)
        push!(candidate_tiers, tier_lookup[loc])
        push!(candidate_adjcosts, float(adjcost_lookup[loc]))
        push!(candidate_preferences, pref)
        push!(candidate_tierdiffs, tierdiff)
        push!(candidate_blockdists, blockdist)
        push!(candidate_scores, 1000.0 * pref + float(adjcost_lookup[loc]))
    end

    candidates = DataFrame(
        Candidate       = 1:length(candidate_locs),
        Location        = candidate_locs,
        Partner         = candidate_partners,
        Block           = candidate_blocks,
        PartnerBlock    = candidate_partnerblks,
        Tier            = candidate_tiers,
        AdjCost         = candidate_adjcosts,
        PreferenceScore = candidate_preferences,
        TierDiff        = candidate_tierdiffs,
        BlockDistance   = candidate_blockdists,
        Score           = candidate_scores
    )

    if use_customer_grouping
        selected_band = "fallback"

        same_block_exact = candidates[(candidates.BlockDistance .== 0) .& (candidates.PreferenceScore .== 0), :]
        if nrow(same_block_exact) > 0
            selected_band = "same/adjacent block exact tier"
            same_block_exact.Score = same_block_exact.AdjCost
            candidates = same_block_exact
        else
            near_candidates = candidates[candidates.BlockDistance .<= 3, :]
            if nrow(near_candidates) > 0
                selected_band = "within 3 blocks"
                near_candidates.Score = 1000.0 .* near_candidates.PreferenceScore .+ 10.0 .* near_candidates.BlockDistance .+ near_candidates.AdjCost
                candidates = near_candidates
            else
                mid_candidates = candidates[(candidates.BlockDistance .> 3) .& (candidates.BlockDistance .<= 5) .& (candidates.TierDiff .<= 1), :]
                if nrow(mid_candidates) > 0
                    selected_band = "3 to 5 blocks (<= 1 tier sacrifice)"
                    mid_candidates.Score = 1000.0 .* mid_candidates.BlockDistance .+ 100.0 .* mid_candidates.TierDiff .+ mid_candidates.AdjCost
                    candidates = mid_candidates
                else
                    far_candidates = candidates[(candidates.BlockDistance .> 5) .& (candidates.BlockDistance .<= 10) .& (candidates.TierDiff .<= 2), :]
                    if nrow(far_candidates) > 0
                        selected_band = "5 to 10 blocks (<= 2 tier sacrifice)"
                        far_candidates.Score = 1000.0 .* far_candidates.BlockDistance .+ 100.0 .* far_candidates.TierDiff .+ far_candidates.AdjCost
                        candidates = far_candidates
                    else
                        veryfar_candidates = candidates[(candidates.BlockDistance .> 10) .& (candidates.TierDiff .== 0), :]
                        if nrow(veryfar_candidates) > 0
                            selected_band = "10+ blocks (exact tier only)"
                            veryfar_candidates.Score = 1000.0 .* veryfar_candidates.BlockDistance .+ veryfar_candidates.AdjCost
                            candidates = veryfar_candidates
                        else
                            selected_band = "historic tier fallback"
                        end
                    end
                end
            end
        end

        println("\nCustomer grouping search band used: ", selected_band)
    end

    println("\nFeasible 40ft candidates:")
    println(sort(candidates, [:Score, :BlockDistance, :PreferenceScore, :AdjCost]))

else
    error("Invalid incoming_width. Use 1 for 20ft or 2 for 40ft.")
end

if nrow(candidates) == 0
    error("No feasible candidates found for the incoming container.")
end

# Yard-specific laden coefficients from regression
laden_coefficients = Dict(
    "One"   => (laden = 1.45787, laden_tier = -0.18356),
    "Two"   => (laden = 0.22452, laden_tier = -0.04366),
    "Three" => (laden = 0.0,     laden_tier = 0.0),      # not significant
    "R"     => (laden = 0.85601, laden_tier = 0.0),      # flat, interaction not significant
    "M"     => (laden = 0.0,     laden_tier = 0.0),      # no data
)

yard_laden = get(laden_coefficients, selected_yard, (laden = 0.0, laden_tier = 0.0))

# Scaling factor = 100 to fit within existing scoring system
# (PreferenceScore ≈ 1000, BlockDistance ≈ 10)
if incoming_laden == 1
    laden_coef = yard_laden.laden
    laden_tier_coef = yard_laden.laden_tier

    if laden_coef == 0.0 && laden_tier_coef == 0.0
        println("\nNo laden penalty for yard ", selected_yard, " (no significant laden effect)")
    else
        scaling_factor = 100.0
        laden_penalty = (laden_coef .+ laden_tier_coef .* candidates.Tier) .* scaling_factor
        candidates.Score .+= laden_penalty

        println("\nYard-specific laden penalty applied for yard ", selected_yard, ":")
        println("  laden coefficient      = ", laden_coef)
        println("  laden:tier coefficient = ", laden_tier_coef)

        if laden_tier_coef == 0.0
            println("  Flat laden penalty added to all candidate tiers = +", round(laden_coef * scaling_factor, digits=2))
        else
            println("  Penalty by tier:")
            for t in sort(unique(candidates.Tier))
                tier_penalty = (laden_coef + laden_tier_coef * t) * scaling_factor
                println("    Tier ", t, ": +", round(tier_penalty, digits=2))
            end
        end
    end
else
    println("\nNo laden penalty (container is empty)")
end


Customer grouping search band used: same block exact tier

Feasible 20ft candidates:
16×9 DataFrame
 Row │ Candidate  Location  Block   Tier   Cost     PreferenceScore  TierDiff  BlockDistance  Score   
     │ Int64      String    String  Int64  Float64  Int64            Int64     Int64          Float64 
─────┼────────────────────────────────────────────────────────────────────────────────────────────────
   1 │        10  30H2      30          2      2.0                0         0              0      2.0
   2 │        79  40AG2     40A         2      2.0                0         0              0      2.0
   3 │        87  41G2      41          2      2.0                0         0              0      2.0
   4 │       142  48F2      48          2      2.0                0         0              0      2.0
   5 │       143  48G2      48          2      2.0                0         0              0      2.0
   6 │       312  69H2      69          2      2.0                0         0   

## 6. Model the Yard Allocation as a MILP

This follows the same style as the class Assignment Problem:
- binary decision variables
- linear objective
- exactly one candidate must be chosen

For a single incoming container, the MILP is simple, but it gives you the same JuMP/HiGHS structure you can later extend to multiple incoming containers.


In [36]:
## PREDICTS SHIFTING FROM YARD-SPECIFIC HISTORIC DATA ##
# ══════════════════════════════════════════════════════ #

function normalize_historic_size(incoming_width::Int)
    if incoming_width == 1
        return 20
    elseif incoming_width == 2
        return 40
    else
        error("incoming_width must be 1 or 2")
    end
end

function safe_string(x)
    if ismissing(x) || x === nothing
        return ""
    else
        return strip(string(x))
    end
end

function safe_float(x, default=0.0)
    s = safe_string(x)
    if s == ""
        return default
    end
    try
        return Float64(x)
    catch
        try
            return parse(Float64, s)
        catch
            return default
        end
    end
end

function safe_int_or_zero(x)
    s = safe_string(x)
    if s == ""
        return 0
    end
    try
        return Int(round(Float64(x)))
    catch
        try
            return parse(Int, s)
        catch
            return 0
        end
    end
end

function location_match_in_history(assigned_loc::String, loc, loclist)
    assigned_loc = strip(assigned_loc)

    if safe_string(loc) == assigned_loc
        return true
    end

    ll = safe_string(loclist)
    if ll == ""
        return false
    end

    parts = [strip(x) for x in split(ll, ",")]
    return assigned_loc in parts
end

function load_historic_data(selected_yard::String)
    historic_folder = joinpath(@__DIR__, "Historic")
    historic_file = joinpath(historic_folder, "$(selected_yard)Historic.csv")

    if !isfile(historic_file)
        error("Historic file not found: " * historic_file)
    end

    hist = CSV.File(historic_file) |> DataFrame
    return hist, historic_file
end

function predict_shift_from_history(
    selected_yard::String,
    assigned_loc::String,
    incoming_eta::Real,
    incoming_width::Int,
    incoming_laden::Int;
    k::Int = 20
    )
    incoming_eta = Float64(incoming_eta)
    
    hist, historic_file = load_historic_data(selected_yard)

    required_cols = ["Location", "LocationList", "ShiftCount", "Laden", "ContainerSize", "Dwell"]
    missing_cols = [c for c in required_cols if !(c in names(hist))]
    if !isempty(missing_cols)
        error("Historic file is missing required columns: $(missing_cols)")
    end

    target_size = normalize_historic_size(incoming_width)

    hist = copy(hist)
    hist[!, :_Location]      = safe_string.(hist.Location)
    hist[!, :_LocationList]  = safe_string.(hist.LocationList)
    hist[!, :_ShiftCount]    = safe_int_or_zero.(hist.ShiftCount)
    hist[!, :_Laden]         = safe_int_or_zero.(hist.Laden)
    hist[!, :_ContainerSize] = safe_int_or_zero.(hist.ContainerSize)
    hist[!, :_Dwell]         = safe_float.(hist.Dwell, NaN)

    filtered = hist[
        (hist._Laden .== incoming_laden) .&
        (hist._ContainerSize .== target_size),
        :
    ]

    match_mask = [
        location_match_in_history(assigned_loc, r._Location, r._LocationList)
        for r in eachrow(filtered)
    ]
    filtered = filtered[match_mask, :]

    filtered = filtered[.!isnan.(filtered._Dwell), :]

    if nrow(filtered) == 0
        return (
            predicted_shift = nothing,
            matched_rows = 0,
            used_rows = 0,
            topk = filtered,
            historic_file = historic_file
        )
    end

    filtered[!, :_dwell_diff] = abs.(filtered._Dwell .- incoming_eta)
    filtered = sort(filtered, :_dwell_diff)

    topk = filtered[1:min(k, nrow(filtered)), :]
    enough_for_estimate = nrow(topk) >= 5
    predicted_shift = enough_for_estimate ? mean(topk._ShiftCount) : nothing

    return (
        predicted_shift = predicted_shift,
        matched_rows = nrow(filtered),
        used_rows = nrow(topk),
        enough_for_estimate = enough_for_estimate,
        topk = topk,
        historic_file = historic_file
    )
end

function print_historic_topk(topk::DataFrame)
    if nrow(topk) == 0
        println("\nNo closest historic containers to display.")
        return
    end

    preferred_cols = [
        "MovementId",
        "ContainerSize",
        "Laden",
        "Location",
        "LocationList",
        "ShiftCount",
        "Dwell",
        "_dwell_diff"
    ]

    cols_to_show = [c for c in preferred_cols if c in names(topk)]

    # println("\nClosest historic containers used for predicted shift:")
    # if isempty(cols_to_show)
    #     println("No matching display columns found.")
    #     println("Available columns are: ", names(topk))  # DEBUG (optional)
    # else
    #     println(select(topk, cols_to_show))
    # end
end

print_historic_topk (generic function with 1 method)

In [37]:
## DEFINES THE OPTIMIZATION MODEL FOR THE YARD ALLOCATION PROBLEM ##
# ═════════════════════════════════════════════════════════════ #

model = Model(HiGHS.Optimizer)

# Number of feasible candidates.
n = nrow(candidates)

# Binary choice variable: X[k] = 1 if candidate k is selected.
@variable(model, X[1:n], Bin)

# Objective: minimize the score.
scores = candidates.Score
@objective(model, Min, sum(scores[k] * X[k] for k in 1:n))

# Exactly one feasible candidate must be chosen.
@constraint(model,
    sum(X[k] for k in 1:n) == 1;
    base_name = "Single Assignment Constraint"
)


Single Assignment Constraint : X[1] + X[2] + X[3] + X[4] + X[5] + X[6] + X[7] + X[8] + X[9] + X[10] + X[11] + X[12] + X[13] + X[14] + X[15] + X[16] = 1

## 7. Solve the MILP using HiGHS


In [38]:
## SOLVES THE OPTIMIZATION MODEL ##
# ═══════════════════════════════ #

set_optimizer_attribute(model, "random_seed", 1)
set_optimizer_attribute(model, "threads", 1)

optimize!(model)


Running HiGHS 1.13.1 (git hash: 1d267d97c): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline 
MIP has 1 row; 16 cols; 16 nonzeros; 16 integer variables (16 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+02, 1e+02]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+00]
Presolving model
1 rows, 16 cols, 16 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-1); columns 0(-16); nonzeros 0(-16) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Cons

In [39]:
## DISPLAYS THE SOLUTION SUMMARY ##
# ═══════════════════════════════ #

println(solution_summary(model))


solution_summary(; result = 1, verbose = false)
├ solver_name          : HiGHS
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ ├ raw_status         : kHighsModelStatusOptimal
│ └ objective_bound    : 1.11075e+02
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : NO_SOLUTION
│ ├ objective_value      : 1.11075e+02
│ ├ dual_objective_value : NaN
│ └ relative_gap         : 0.00000e+00
└ Work counters
  ├ solve_time (sec)   : 3.82917e-04
  ├ simplex_iterations : 0
  ├ barrier_iterations : -1
  └ node_count         : 0


## 8. Extract and Display the Selected Placement

As in the class MILP notebook, we check the optimization status and then extract the binary solution.


In [40]:
model_status = termination_status(model)

if model_status === MOI.OPTIMAL
    println("The solver found an optimal solution.\n")
    println("Historical target tier = ", target_tier)
    println("Customer grouping active = ", use_customer_grouping)

    X_OPT = value.(X)
    X_OPT = sparse(X_OPT)
    X_OPT = round.(Int, X_OPT)

    I, _ = findnz(X_OPT)

    if incoming_width == 1
        for k in I
            chosen = candidates[k, :]
            assigned_location = string(chosen.Location)

            println("Selected 20ft slot:")
            println("  - Location        : ", assigned_location)
            println("  - Block           : ", chosen.Block)
            println("  - Tier            : ", chosen.Tier)
            println("  - Cost            : ", chosen.Cost)
            println("  - BlockDistance   : ", chosen.BlockDistance)
            println("  - PreferenceScore : ", chosen.PreferenceScore)
            println("  - TierDiff        : ", chosen.TierDiff)
            println("  - Score           : ", chosen.Score)

            historic_prediction = predict_shift_from_history(
                selected_yard,
                assigned_location,
                incoming_eta,
                incoming_width,
                incoming_laden
            )

            println("\nHistoric shift prediction:")
            println("  - Historic file   : ", historic_prediction.historic_file)
            println("  - Matched rows    : ", historic_prediction.matched_rows)
            println("  - Used closest    : ", historic_prediction.used_rows)

            if historic_prediction.used_rows == 0
                println("  - Predicted shift : no matching historic rows found")
            elseif !historic_prediction.enough_for_estimate
                println("  - Predicted shift : Not enough containers found to make accurate estimate")
                print_historic_topk(historic_prediction.topk)
            else
                println("  - Predicted shift : ", round(historic_prediction.predicted_shift, digits=3))
                println("Average historical shifting  = ", avg_reshifting)
                print_historic_topk(historic_prediction.topk)

            end
        end
    else
        for k in I
            chosen = candidates[k, :]
            assigned_location = string(chosen.Location)

            println("Selected 40ft slot pair:")
            println("  - Location        : ", assigned_location)
            println("  - Partner         : ", chosen.Partner)
            println("  - Block           : ", chosen.Block)
            println("  - PartnerBlock    : ", chosen.PartnerBlock)
            println("  - Tier            : ", chosen.Tier)
            println("  - AdjCost         : ", chosen.AdjCost)
            println("  - BlockDistance   : ", chosen.BlockDistance)
            println("  - PreferenceScore : ", chosen.PreferenceScore)
            println("  - TierDiff        : ", chosen.TierDiff)
            println("  - Score           : ", chosen.Score)

            historic_prediction = predict_shift_from_history(
                selected_yard,
                assigned_location,
                incoming_eta,
                incoming_width,
                incoming_laden
            )

            println("\nHistoric shift prediction:")
            println("  - Historic file   : ", historic_prediction.historic_file)
            println("  - Matched rows    : ", historic_prediction.matched_rows)
            println("  - Used closest    : ", historic_prediction.used_rows)

            if historic_prediction.used_rows == 0
                println("  - Predicted shift : no matching historic rows found")
            elseif !historic_prediction.enough_for_estimate
                println("  - Predicted shift : Not enough containers for an accurate prediction")
                print_historic_topk(historic_prediction.topk)
            else
                println("  - Predicted shift : ", round(historic_prediction.predicted_shift, digits=3))
                println("Average historical shifting  = ", avg_reshifting)
                print_historic_topk(historic_prediction.topk)

            end
        end
    end

else
    println("The solver did not find an optimal solution.")
    println("Termination status: ", model_status)
end


The solver found an optimal solution.

Historical target tier = 2
Customer grouping active = true
Selected 20ft slot:
  - Location        : 67SB2
  - Block           : 67S
  - Tier            : 2
  - Cost            : 2.0
  - BlockDistance   : 0
  - PreferenceScore : 0
  - TierDiff        : 0
  - Score           : 111.07499999999999

Historic shift prediction:
  - Historic file   : /Users/sheil/Desktop/SUTD/Y2/T4/40.011 Data and Business Analytics/Poh Tiong Choon/DBA PTC Multi Yard/Universal/Historic/OneHistoric.csv
  - Matched rows    : 14
  - Used closest    : 14
  - Predicted shift : 0.857
Average historical shifting  = 0.7


## 10. Interpretation

This version uses:
- historical ETA and ShiftCount to infer a target tier
- customer containers currently in the yard to infer preferred nearby blocks
- a block-aware candidate search policy before the MILP is solved

The MILP still chooses exactly one feasible slot or slot pair, but the candidate scores now reflect both:
1. the historically preferred tier, and
2. proximity to the customer's existing blocks in the yard.
